In [1]:
import sys
import os
import warnings
import torch
import requests
import json
import csv
from esm import Alphabet, FastaBatchedDataset, pretrained
import shutil

warnings.filterwarnings("ignore")
sys.path.append("./src/")
from CLEAN.infer import *
# pre-compute esm-1b embedding for sequences in new-392 dataset
ensure_dirs("data/esm_data")
ensure_dirs("data/pretrained")
from CLEAN.infer import *


/home/wuke/anaconda3/envs/AIGEM/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def fasta_to_csv(fasta_name, csv_name):
    with open(fasta_name, 'r') as fastafile, open(csv_name, 'w') as outfile:
        csv_rows = []
        fasta_id = ''
        for line in fastafile:
            line = line.strip()
            if line.startswith('>'):
                if fasta_id:
                    csv_rows.append([fasta_id, fasta_sequence])
                fasta_id = line[1:].split()[0]  # 只保留ID部分
                fasta_sequence = ''
            else:
                fasta_sequence += line
        if fasta_id:
            csv_rows.append([fasta_id, fasta_sequence])
        header = ['Entry', 'Sequence']
        csv_rows.insert(0, header)
        csvwriter = csv.writer(outfile, delimiter='\t')
        csvwriter.writerows(csv_rows)

def csv_to_fasta(csv_name, fasta_name):
    df = pd.read_csv(csv_name, delimiter='\t')  # 读取 CSV 文件
    with open(fasta_name, 'w') as fasta_out:
        for _, row in df.iterrows():
            entry = row['Entry']
            sequence = row['Sequence'].replace('\n', '').replace('\r', '')  # 确保去掉换行符
            fasta_out.write(">" + entry + "\n" + sequence + "\n")  # 用字符串拼接替代 f-string

def get_clean_predict_result(target_protein_result_file,clean_result_file):
    data = []
    max_columns = 0
    with open(target_protein_result_file, 'r', encoding='utf-8') as file:
        reader = csv.reader(file)
        for row in reader:
            # 获取当前行的最大列数
            max_columns = max(max_columns, len(row))
            data.append(row)
    for row in data:
        row.extend([''] * (max_columns - len(row)))

    tmp_res = pd.DataFrame(data)
    for i in range(1,max_columns):
        tmp_res[i] = tmp_res[i].apply(lambda x:x.split('/')[0].split(':')[1] if '/' in x else x)
    tmp_res

    clean_result = {}
    for index,row in tmp_res.iterrows():
        ec_list = []
        for i in range(1,max_columns):
            ec_list.append(row[i])
        clean_result[row[0]] = [x for x in ec_list if x != '']
        
    with open(clean_result_file, 'w') as json_file:
        json.dump(clean_result, json_file) 

In [3]:
target_protein_fasta_folder = '/home/wuke/project/microbiome–drug/Data/AGORA2_genome/'

genome_files = []
# 遍历指定路径下的所有文件和子目录
for root, dirs, files in os.walk(target_protein_fasta_folder):
    for file in files:
        if file.endswith('.fasta'):
            genome_files.append(file.replace('.fasta',''))
# 打印结果
print(genome_files)
print(len(genome_files))

['GCF_000339595.1_ASM33959v1', 'GCF_000391805.1_Ente_faec_7330381-1_V1', 'GCF_000006765.1_ASM676v1', 'GCF_000196015.1_ASM19601v1', 'GCF_000174535.1_ASM17453v1', 'GCF_000161275.1_ASM16127v1', 'GCF_000259315.1_ASM25931v1', 'GCF_000251485.1_ASM25148v2', 'GCF_000022345.1_ASM2234v1', 'GCF_000204805.1_ASM20480v2', 'GCF_000161875.1_ASM16187v1', 'GCF_000329665.1_ASM32966v1', 'GCF_000359785.1_Stap_aure_HI013_V1', 'GCF_000300955.1_Kleb_pneu_WGLW5_V1', 'GCF_000303955.2_ASM30395v2', 'GCF_000320035.1_Esch_coli_11-02092_V1', 'GCF_000233395.1_ASM23339v2', 'GCF_000371425.1_Clos_inno_2959_V1', 'GCF_000262855.1_VRSA03_3', 'GCF_000249075.1_ASM24907v2', 'GCF_000174555.1_ASM17455v1', 'GCF_000289655.1_ASM28965v1', 'GCF_000290095.1_ASM29009v1', 'GCF_000369345.1_Acin_baum_NIPH_80_V1', 'GCF_000331085.3_ASM33108v2', 'GCF_000345305.2_ASM34530v2', 'GCF_000252805.1_ASM25280v2', 'GCF_000254575.1_ASM25457v2', 'GCF_000260475.1_NCCP15658v1.0', 'GCF_000249335.1_ASM24933v2', 'GCF_000285755.1_ASM28575v1', 'GCF_000270065.

In [4]:
if 'GCF_001543345.1_ASM154334v1' in genome_files:
    print(1)

1


In [ ]:
target_protein_fasta_folder = '/home/wuke/project/microbiome–drug/Data/AGORA2_genome/'

genome_files = []
# 遍历指定路径下的所有文件和子目录
for root, dirs, files in os.walk(target_protein_fasta_folder):
    for file in files:
        if file.endswith('.fasta'):
            genome_files.append(file.replace('.fasta',''))
# 打印结果
print(genome_files)
print(len(genome_files))
# genome_files =      ['GCF_002025805.1_ASM202580v1','ERR1022470_2','GCF_000789395.1_ASM78939v1']

for file_name in tqdm(genome_files):
    # print(file_name)
    target_protein_fasta_file = target_protein_fasta_folder + file_name + '.fasta'
    target_protein_csv_file = './data/' + file_name + '.csv'
    target_protein_result_file = './results/'+ file_name +'_maxsep.csv'
    clean_result_file = './results/'+ file_name +'_clean_result.json'  
    destination_file = './data/' + file_name + '.fasta'
    if not os.path.exists(clean_result_file):
        try:
            fasta_to_csv(target_protein_fasta_file, target_protein_csv_file)
            csv_to_fasta(target_protein_csv_file, destination_file)
            retrive_esm1b_embedding(file_name)
            infer_maxsep("split100",file_name, report_metrics=True, pretrained=True)
        except:
            if os.path.exists(target_protein_csv_file):
                os.remove(target_protein_csv_file)
            if os.path.exists(destination_file):        
                os.remove(destination_file)
            print('error:',file_name)
            continue
        get_clean_predict_result(target_protein_result_file,clean_result_file)
        os.remove(destination_file)

['GCF_000339595.1_ASM33959v1', 'GCF_000391805.1_Ente_faec_7330381-1_V1', 'GCF_000006765.1_ASM676v1', 'GCF_000196015.1_ASM19601v1', 'GCF_000174535.1_ASM17453v1', 'GCF_000161275.1_ASM16127v1', 'GCF_000259315.1_ASM25931v1', 'GCF_000251485.1_ASM25148v2', 'GCF_000022345.1_ASM2234v1', 'GCF_000204805.1_ASM20480v2', 'GCF_000161875.1_ASM16187v1', 'GCF_000329665.1_ASM32966v1', 'GCF_000359785.1_Stap_aure_HI013_V1', 'GCF_000300955.1_Kleb_pneu_WGLW5_V1', 'GCF_000303955.2_ASM30395v2', 'GCF_000320035.1_Esch_coli_11-02092_V1', 'GCF_000233395.1_ASM23339v2', 'GCF_000371425.1_Clos_inno_2959_V1', 'GCF_000262855.1_VRSA03_3', 'GCF_000249075.1_ASM24907v2', 'GCF_000174555.1_ASM17455v1', 'GCF_000289655.1_ASM28965v1', 'GCF_000290095.1_ASM29009v1', 'GCF_000369345.1_Acin_baum_NIPH_80_V1', 'GCF_000331085.3_ASM33108v2', 'GCF_000345305.2_ASM34530v2', 'GCF_000252805.1_ASM25280v2', 'GCF_000254575.1_ASM25457v2', 'GCF_000260475.1_NCCP15658v1.0', 'GCF_000249335.1_ASM24933v2', 'GCF_000285755.1_ASM28575v1', 'GCF_000270065.

  0%|          | 0/48 [00:00<?, ?it/s]

error: GCF_900083545.1_PRJEB13794
error: GCF_000384195.1_ASM38419v1
error: GCF_002221705.2_ASM222170v2
error: GCF_000404205.1_ASM40420v1
Transferred model to GPU
Read data/GCF_000225685.1_Erys_bact_2_2_44A_V1.fasta with 4709 sequences
Processing 1 of 357 batches (67 sequences)
Processing 2 of 357 batches (62 sequences)
Processing 3 of 357 batches (58 sequences)
Processing 4 of 357 batches (55 sequences)
Processing 5 of 357 batches (52 sequences)
Processing 6 of 357 batches (49 sequences)
Processing 7 of 357 batches (47 sequences)
Processing 8 of 357 batches (45 sequences)
Processing 9 of 357 batches (43 sequences)
Processing 10 of 357 batches (41 sequences)
Processing 11 of 357 batches (40 sequences)
Processing 12 of 357 batches (39 sequences)
Processing 13 of 357 batches (38 sequences)
Processing 14 of 357 batches (37 sequences)
Processing 15 of 357 batches (36 sequences)
Processing 16 of 357 batches (35 sequences)
Processing 17 of 357 batches (34 sequences)
Processing 18 of 357 batch

100%|██████████| 5242/5242 [00:00<00:00, 24764.78it/s]


Calculating eval distance map, between 4709 test ids and 5242 train EC cluster centers


...

100%|██████████| 5242/5242 [00:06<00:00, 770.34it/s] 


Calculating eval distance map, between 2783 test ids and 5242 train EC cluster centers


2783it [00:03, 827.36it/s]


############ EC calling results using maximum separation ############
---------------------------------------------------------------------------
>>> total samples: 2783 | total ec: 2783 
>>> precision: 0.0 | recall: 0.0| F1: 0.0 | AUC: 0.5 
---------------------------------------------------------------------------


100%|██████████| 48/48 [1:30:05<00:00, 112.62s/it]
